# Data preparation

## Processing Guerre et Paix

In [14]:
import subprocess
import sys

subprocess.check_call([sys.executable, "-m", "pip", "install", "ebooklib", "beautifulsoup4", "-q"])

import re
import unicodedata
from pathlib import Path
from ebooklib import epub
import ebooklib
from bs4 import BeautifulSoup

# ─── Configuration ───────────────────────────────────────────────────────────

BASE_DIR = Path("guerre_et_paix")
EPUB_FILES = [
    "Guerre et Paix T1.epub",
    "Guerre et Paix T2.epub",
    "Guerre et Paix T3.epub",
]

# Titles matching these are metadata/boilerplate, not chapters
SKIP_EXACT = {'FIN', 'NOTES:'}
SKIP_SUBSTRINGS = [
    'GUTENBERG', 'TRADUCTION', 'AVANT TILSITT', "L'INVASION",
    'BORODINO', 'COMTE L', 'FIN DU', 'THE FULL',
]

# ─── Helpers ─────────────────────────────────────────────────────────────────

def slugify(title: str) -> str:
    """Convert a title to a safe filesystem name (strips accents, replaces spaces)."""
    title = unicodedata.normalize('NFD', title)
    title = ''.join(c for c in title if not unicodedata.combining(c))
    title = re.sub(r'[^\w\s]', '', title)
    title = re.sub(r'\s+', '_', title.strip())
    return title


def should_skip(title: str) -> bool:
    t = title.strip().upper()
    if t in SKIP_EXACT:
        return True
    return any(kw in t for kw in SKIP_SUBSTRINGS)


def get_epub_item_content(book, filename: str) -> bytes:
    """Return the raw HTML bytes for the epub item whose name ends with filename."""
    for item in book.get_items_of_type(ebooklib.ITEM_DOCUMENT):
        if item.get_name().endswith(filename) or filename in item.get_name():
            return item.get_content()
    return b''


def parse_href(href: str):
    """Split 'file.xhtml#anchor' into ('file.xhtml', 'anchor')."""
    if href and '#' in href:
        return href.rsplit('#', 1)
    return href, None


def extract_text_between_anchors(html_bytes: bytes, start_id: str, end_id: str = None) -> str:
    """
    Extract text from the element with id=start_id up to (not including)
    the element with id=end_id.
    Handles anchors that are headings (<h3 id="...">) or embedded inside
    text elements (<p><a id="...">).
    """
    soup = BeautifulSoup(html_bytes, 'html.parser')
    TEXT_TAGS = {'p', 'h1', 'h2', 'h3', 'h4', 'h5', 'h6', 'blockquote'}

    collecting = False
    paragraphs = []
    seen = set()  # object ids of already-collected tags (avoids duplicates from nesting)

    for tag in soup.find_all(True):
        tag_id = tag.get('id', '')

        # Stop before the next chapter's anchor
        if collecting and end_id and tag_id == end_id:
            break

        # Start collecting when we reach the chapter anchor.
        # Also handles <p><a id="anchor"> via find() on the parent text element.
        if not collecting:
            if tag_id == start_id:
                collecting = True
            elif tag.name in TEXT_TAGS and tag.find(id=start_id):
                collecting = True

        if collecting and tag.name in TEXT_TAGS and id(tag) not in seen:
            # Skip nested text tags whose parent was already collected
            if not any(id(p) in seen for p in tag.parents if p.name in TEXT_TAGS):
                text = tag.get_text(separator=' ', strip=True)
                if text:
                    paragraphs.append(text)
                seen.add(id(tag))

    return '\n\n'.join(paragraphs)


# ─── TOC parsing ─────────────────────────────────────────────────────────────

def build_chapter_list(toc) -> list:
    """
    Walk the epub TOC and return a flat list of
        (path_parts: list[str], chapter_name: str, href: str)
    where path_parts encodes the hierarchy TOME > PARTIE > CHAPITRE.
    """
    chapters = []
    state = {
        'tome': None,    'tome_i': 0,
        'partie': None,  'partie_i': 0,
        'chapitre': None,'chapitre_i': 0,
        'ch_i': 0,
    }

    def process(items):
        for item in items:
            if isinstance(item, tuple):
                section, children = item
                title = (section.title or '').strip()
                t = title.upper()

                if 'TOME' in t:
                    state['tome_i'] += 1
                    state['partie_i'] = state['chapitre_i'] = 0
                    state['tome'] = f"{state['tome_i']:02d}_{slugify(title)}"
                    state['partie'] = None
                    # Children of TOME are metadata links — do not recurse

                elif 'PARTIE' in t:
                    state['partie_i'] += 1
                    state['chapitre_i'] = 0
                    state['partie'] = f"{state['partie_i']:02d}_{slugify(title)}"
                    # Children of PARTIE are subtitle links — do not recurse

                elif 'CHAPITRE' in t or 'CHAPTER' in t:
                    state['chapitre_i'] += 1
                    state['ch_i'] = 0
                    state['chapitre'] = f"{state['chapitre_i']:02d}_{slugify(title)}"
                    process(children)  # children are the actual chapter links

                else:
                    process(children)

            elif isinstance(item, epub.Link):
                title = (item.title or '').strip()
                if should_skip(title) or not item.href or not state['chapitre']:
                    continue
                state['ch_i'] += 1
                ch_name = f"{state['ch_i']:02d}_{slugify(title)}"
                path = [p for p in [state['tome'], state['partie'], state['chapitre']] if p]
                chapters.append((path, ch_name, item.href))

    process(toc)
    return chapters


# ─── Main processing ─────────────────────────────────────────────────────────

for epub_file in EPUB_FILES:
    epub_path = BASE_DIR / epub_file
    book_name = epub_file[:-5]   # strip .epub
    book_dir  = BASE_DIR / book_name

    print(f"\n{'='*60}")
    print(f"Processing: {epub_file}")

    book = epub.read_epub(str(epub_path))
    html_cache: dict = {}

    def get_html(filename: str) -> bytes:
        if filename not in html_cache:
            html_cache[filename] = get_epub_item_content(book, filename)
        return html_cache[filename]

    chapters = build_chapter_list(book.toc)
    print(f"Chapters found: {len(chapters)}")

    for i, (path_parts, ch_name, href) in enumerate(chapters):
        next_href = chapters[i + 1][2] if i + 1 < len(chapters) else None

        curr_file, curr_anchor = parse_href(href)
        next_file, next_anchor = parse_href(next_href) if next_href else (None, None)

        # Build output directory: book_dir / TOME / PARTIE / CHAPITRE / chapter_name
        chapter_dir = book_dir
        for part in path_parts:
            chapter_dir = chapter_dir / part
        chapter_dir = chapter_dir / ch_name
        chapter_dir.mkdir(parents=True, exist_ok=True)

        # Use next_anchor as a boundary only when both chapters share the same HTML file
        end_anchor = next_anchor if (next_file == curr_file) else None

        text = ''
        if curr_anchor:
            text = extract_text_between_anchors(get_html(curr_file), curr_anchor, end_anchor)

        txt_path = chapter_dir / f"{ch_name}.txt"
        txt_path.write_text(text, encoding='utf-8')

    print(f"Saved to: {book_dir}/")

print("\nDone!")



Processing: Guerre et Paix T1.epub
Chapters found: 104
Saved to: guerre_et_paix\Guerre et Paix T1/

Processing: Guerre et Paix T2.epub
Chapters found: 102
Saved to: guerre_et_paix\Guerre et Paix T2/

Processing: Guerre et Paix T3.epub
Chapters found: 134
Saved to: guerre_et_paix\Guerre et Paix T3/

Done!


## Processing Les Miserables

In [2]:
import subprocess
import sys

subprocess.check_call([sys.executable, "-m", "pip", "install", "ebooklib", "beautifulsoup4", "-q"])

import re
import unicodedata
from pathlib import Path
from ebooklib import epub
import ebooklib
from bs4 import BeautifulSoup

# ─── Configuration ───────────────────────────────────────────────────────────

BASE_DIR = Path("les_miserables")
EPUB_FILES = sorted(f.name for f in BASE_DIR.glob("*.epub"))

# Top-level links to skip (not chapters)
SKIP_TITLES = {'Page titre', '\u00c0 propos de cette \u00e9dition \u00e9lectronique'}

# ─── Helpers ─────────────────────────────────────────────────────────────────

def slugify_lm(title: str, max_len: int = 40) -> str:
    """Convert a title to a safe filesystem name, truncated to max_len.
    max_len=40 keeps total paths under Windows MAX_PATH (260 chars).
    """
    title = unicodedata.normalize('NFD', str(title))
    title = ''.join(c for c in title if not unicodedata.combining(c))
    title = re.sub(r'[^\w\s]', '', title)
    title = re.sub(r'\s+', '_', title.strip())
    return title[:max_len]


def get_epub_item_lm(book, filename: str) -> bytes:
    """Return the raw bytes of an epub item by filename."""
    for item in book.get_items_of_type(ebooklib.ITEM_DOCUMENT):
        name = item.get_name()
        if name.endswith(filename) or name.endswith('/' + filename):
            return item.get_content()
    return b''


def extract_all_text_lm(html_bytes: bytes) -> str:
    """Extract all paragraph/heading text from an HTML file (no anchor slicing needed)."""
    soup = BeautifulSoup(html_bytes, 'html.parser')
    TEXT_TAGS = {'p', 'h1', 'h2', 'h3', 'h4', 'h5', 'h6', 'blockquote'}
    seen = set()
    paragraphs = []

    for tag in soup.find_all(TEXT_TAGS):
        if id(tag) in seen:
            continue
        if any(id(p) in seen for p in tag.parents if p.name in TEXT_TAGS):
            continue
        text = tag.get_text(separator=' ', strip=True)
        if text:
            paragraphs.append(text)
        seen.add(id(tag))

    return '\n\n'.join(paragraphs)


# ─── TOC parsing ─────────────────────────────────────────────────────────────

def build_chapter_list_lm(toc) -> list:
    """
    Parse the Les Miserables epub TOC and return a flat list of
        (livre_folder: str, chapitre_folder: str, href: str).
    Structure: LIVRE (section) -> CHAPITRE (link with full title).
    """
    chapters = []
    livre_i = 0

    for item in toc:
        if isinstance(item, tuple):
            section, children = item
            livre_title = (section.title or '').strip()
            livre_i += 1
            ch_i = 0
            livre_folder = f"{livre_i:02d}_{slugify_lm(livre_title)}"

            for child in children:
                if isinstance(child, epub.Link):
                    ch_title = (child.title or '').strip()
                    if ch_title in SKIP_TITLES or not child.href:
                        continue
                    ch_i += 1
                    ch_folder = f"{ch_i:02d}_{slugify_lm(ch_title)}"
                    chapters.append((livre_folder, ch_folder, child.href))

        elif isinstance(item, epub.Link):
            # Standalone top-level links (Page titre, A propos...) -- skip
            pass

    return chapters


# ─── Main processing ─────────────────────────────────────────────────────────

for epub_file in EPUB_FILES:
    epub_path = BASE_DIR / epub_file
    book_name = epub_file[:-5]   # strip .epub
    book_dir  = BASE_DIR / book_name

    print(f"\n{'='*60}")
    print(f"Processing: {epub_file}")

    book = epub.read_epub(str(epub_path))
    html_cache_lm: dict = {}

    def get_html_lm(filename: str) -> bytes:
        if filename not in html_cache_lm:
            html_cache_lm[filename] = get_epub_item_lm(book, filename)
        return html_cache_lm[filename]

    chapters = build_chapter_list_lm(book.toc)
    print(f"Chapters found: {len(chapters)}")

    for livre_folder, ch_folder, href in chapters:
        chapter_dir = book_dir / livre_folder / ch_folder
        chapter_dir.mkdir(parents=True, exist_ok=True)

        text = extract_all_text_lm(get_html_lm(href))

        txt_path = chapter_dir / f"{ch_folder}.txt"
        txt_path.write_text(text, encoding='utf-8')

    print(f"Saved to: {book_dir}/")

print("\nDone!")



Processing: hugo_les_miserables_cosette.epub
Chapters found: 76
Saved to: les_miserables\hugo_les_miserables_cosette/

Processing: hugo_les_miserables_fantine.epub
Chapters found: 70
Saved to: les_miserables\hugo_les_miserables_fantine/

Processing: hugo_les_miserables_idylle_plumet_epopee_st_denis.epub
Chapters found: 76
Saved to: les_miserables\hugo_les_miserables_idylle_plumet_epopee_st_denis/

Processing: hugo_les_miserables_jean_valjean.epub
Chapters found: 67
Saved to: les_miserables\hugo_les_miserables_jean_valjean/

Processing: hugo_les_miserables_marius.epub
Chapters found: 76
Saved to: les_miserables\hugo_les_miserables_marius/

Done!


## Setting up the Propp pipeline

In [13]:
def process_book(book_path, spacy_model, mentions_detection_model, coreference_resolution_model):

    import torch
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    ######################################################################################

    from propp_fr import load_text_file
    text_content = load_text_file(book_path)

    ######################################################################################

    from propp_fr import generate_tokens_df
    tokens_df = generate_tokens_df(text_content, spacy_model)

    ######################################################################################

    from propp_fr import load_tokenizer_and_embedding_model, get_embedding_tensor_from_tokens_df

    # Load the tokenizer and pre-trained embedding model
    tokenizer, embedding_model = load_tokenizer_and_embedding_model(
        mentions_detection_model["base_model_name"],
      )

    # Generate embeddings for all tokens
    tokens_embedding_tensor = get_embedding_tensor_from_tokens_df(
        text_content,
        tokens_df,
        tokenizer,
        embedding_model,
        device=device,
      )

    ######################################################################################

    from propp_fr import generate_entities_df

    entities_df = generate_entities_df(
        tokens_df,
        tokens_embedding_tensor,
        mentions_detection_model,
    )

    ######################################################################################

    from propp_fr import add_features_to_entities

    entities_df = add_features_to_entities(entities_df, tokens_df)

    ######################################################################################

    from propp_fr import perform_coreference

    entities_df = perform_coreference(
        entities_df,
        tokens_embedding_tensor,
        coreference_resolution_model,
        )

    ######################################################################################

    from propp_fr import extract_attributes
    tokens_df = extract_attributes(entities_df, tokens_df)

    ######################################################################################

    from propp_fr import generate_characters_dict

    characters_dict = generate_characters_dict(tokens_df, entities_df)

    ######################################################################################

    return tokens_df, entities_df, characters_dict

### Propp test run

In [ ]:
import os
import gc
import datetime
import traceback
import torch
from propp_fr import load_models, save_tokens_df, save_entities_df, save_book_file

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Загружаем модели один раз для всех книг.
# Если GPU недоступен, propp_fr загружает модели с pickle без map_location='cpu',
# поэтому временно подменяем torch.load чтобы принудительно использовать CPU.
if not torch.cuda.is_available():
    _original_torch_load = torch.load
    def _cpu_torch_load(f, map_location=None, **kwargs):
        return _original_torch_load(f, map_location=map_location or 'cpu', **kwargs)
    torch.load = _cpu_torch_load
    try:
        spacy_model, mentions_detection_model, coreference_resolution_model = load_models()
    finally:
        torch.load = _original_torch_load
else:
    spacy_model, mentions_detection_model, coreference_resolution_model = load_models()

for subdir, dirs, files in os.walk("guerre_et_paix/Guerre et Paix T1/01_TOME_I/01_PREMIERE_PARTIE/01_CHAPITRE_PREMIER/"):
    for file in files:
        if not file.endswith(".txt"):
            continue

        book_name = file[:-4]
        book_path = os.path.join(subdir, file)

        # Пропускаем уже обработанные файлы
        book_file_path = os.path.join(subdir, book_name + "_book_file.book")
        if os.path.exists(book_file_path):
            print(f"Skipping (already processed): {file}")
            continue

        print("####################################################################################")
        print(f"Processing file: {file}")
        print(f"Start time: {datetime.datetime.now()}")

        try:
            tokens, entities, characters = process_book(
                book_path, spacy_model, mentions_detection_model, coreference_resolution_model
            )
            save_tokens_df(tokens, book_name + "_tokens", subdir)
            save_entities_df(entities, book_name + "_entities", subdir)
            save_book_file(characters, book_name + "_book_file", subdir)
            print(f"End time: {datetime.datetime.now()}")
        except Exception as e:
            print(f"ERROR processing {file}: {e}")
            traceback.print_exc()
        finally:
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

Using device: cpu
Loading models...
CPU version detected. No GPU required. Skipping installation.
Loaded Spacy Model: fr_dep_news_trf
CUDA is not available, model will run on CPU.
CUDA is not available, Spacy model will run on CPU.
Model Downloaded Successfully
Saving model locally to: C:\Users\kovre\PycharmProjects\balzac_characters\AntoineBourgois\propp-fr_NER_camembert-large_FAC_GPE_LOC_PER_TIME_VEH
Model Downloaded Successfully
Saving model locally to: C:\Users\kovre\PycharmProjects\balzac_characters\AntoineBourgois\propp-fr_coreference-resolution_camembert-large_PER

Models Loaded Successfully:
Spacy: fr_dep_news_trf
Mentions Detection: AntoineBourgois/propp-fr_NER_camembert-large_FAC_GPE_LOC_PER_TIME_VEH
Coreference Resolution: AntoineBourgois/propp-fr_coreference-resolution_camembert-large_PER
####################################################################################
Processing file: 01_I.txt
Start time: 2026-04-12 14:14:00.388686


C:\Users\kovre\PycharmProjects\balzac_characters\.venv\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\kovre\.cache\huggingface\hub\models--almanach--camembert-large. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling 

In [2]:
from propp_fr import load_tokens_df

tokens_df = load_tokens_df("balzac-55-FC-tenebreuse-affaire_tokens", "la_comedie_humaine/etudes_des_moeurs/scenes_de_la_vie_politique/balzac-55-FC-tenebreuse-affaire")

print(tokens_df.columns.tolist())

['paragraph_ID', 'sentence_ID', 'token_ID_within_sentence', 'token_ID_within_document', 'word', 'lemma', 'byte_onset', 'byte_offset', 'POS_tag', 'dependency_relation', 'syntactic_head_ID', 'morph', 'is_mention_head', 'char_att_agent', 'char_att_patient', 'char_att_mod', 'char_att_poss']


In [3]:
from propp_fr import load_book_file

characters_dict = load_book_file("balzac-55-FC-tenebreuse-affaire_book_file", "la_comedie_humaine/etudes_des_moeurs/scenes_de_la_vie_politique/balzac-55-FC-tenebreuse-affaire")

# print((characters_dict))



C:\Users\covre\PycharmProjects\balzac_comedie_humaine\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


propp_fr package loaded successfully.


In [25]:
print(characters_dict[0]['mentions']['proper'])


[{'n': 'michu', 'c': 131}, {'n': 'grévin', 'c': 18}, {'n': 'le bonhomme d’ hauteserre', 'c': 2}, {'n': 'd’ hauteserre', 'c': 2}, {'n': 'ci - devant marquis de chargebœuf', 'c': 1}, {'n': 'du chef de la maison de chargebœuf', 'c': 1}, {'n': 'monsieur de chargebœuf', 'c': 1}, {'n': 'le sénateur de l’ empire', 'c': 1}, {'n': 'le vieux marquis de chargebœuf', 'c': 1}, {'n': 'mon pauvre michu', 'c': 1}, {'n': 'le marquis de chargebœuf', 'c': 1}, {'n': 'un homme entièrement semblable à michu', 'c': 1}, {'n': 'monsieur pigoult', 'c': 1}, {'n': 'monsieur d’ hauteserre', 'c': 1}, {'n': 'le marquis de simeuse', 'c': 1}, {'n': 'au marquis de simeuse', 'c': 1}, {'n': 'cinq - cygne', 'c': 1}, {'n': 'l’ aîné des d’ hauteserre', 'c': 1}, {'n': 'des d’ hauteserre', 'c': 1}, {'n': 'adrien', 'c': 1}, {'n': 'l’ ancien piqueur de la maison de simeuse', 'c': 1}, {'n': 'beauvisage', 'c': 1}, {'n': 'l’ ancien régisseur de gondreville', 'c': 1}, {'n': 'l’ abbé goujet', 'c': 1}, {'n': 'de michu', 'c': 1}]
